# voice-rs — huấn luyện adapter trên Colab (CPU)

Notebook này: cài Rust → clone repo → tải asset nặng (GitHub Release `assets-v1`)
→ export trọng số `dec` → sinh features ContentVec → huấn luyện **chỉ `Adapter`**
với `dec` đóng băng.

Chạy trong **Runtime → Change runtime type → CPU** (bản này chưa hỗ trợ GPU).

⚠️ chi phí: Colab free ~2 core. `SEG=86` (1 s) ≈ 1.5–2 h cho 250 bước.
`SEG` = số frame latent; tăng lên sẽ lâu hơn.


## 1. Cài Rust


In [ ]:
%%bash
set -e
if ! command -v cargo >/dev/null 2>&1; then
  curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
fi
source $HOME/.cargo/env
cargo --version && rustc --version


## 2. Clone repo


In [ ]:
%%bash
set -e
cd /content
[ -d voice-rs ] || git clone https://github.com/hungpham10/voice-rs.git
cd voice-rs && git log --oneline -1 && ls


## 3. Build workspace (release)
Lần đầu mất vài phút (Burn + deps).


In [ ]:
%%bash
set -e
cd /content/voice-rs
source $HOME/.cargo/env
cargo build --release -p voice-models -p voice-hubert -p voice-train


## 4. Tải asset nặng từ GitHub Release `assets-v1`
`pretrained_ljs.pth` (139 MB) + ContentVec ONNX (360 MB).


In [ ]:
%%bash
set -e
cd /content/voice-rs
mkdir -p sources/features sources/onnx_hubert_new
BASE=https://github.com/hungpham10/voice-rs/releases/download/assets-v1
wget -q --show-progress -O sources/pretrained_ljs.pth      $BASE/pretrained_ljs.pth
wget -q --show-progress -O sources/onnx_hubert_new/model.onnx $BASE/model.onnx
ls -lh sources/pretrained_ljs.pth sources/onnx_hubert_new/model.onnx


## 5. Export trọng số `dec` (offline, PyTorch)
Colab đã có `torch`. Script cũng tự kiểm **tính chất vật lý** (RMS đầu ra phải
ở mức tiếng nói — chặn bug gộp weight-norm sai trục, xem §2.3.1).


In [ ]:
%%bash
set -e
cd /content/voice-rs
python -m pip -q install numpy
python scripts/export_vits_dec.py --hps scripts/ljs_base.json


## 6. Sinh features ContentVec cho wav đích
`samples/422-122949-0031.wav` (LibriSpeech, 16 kHz) → `.npy [1,T,768]`.


In [ ]:
%%bash
set -e
cd /content/voice-rs
source $HOME/.cargo/env
cargo run --release -p voice-hubert --example embed -- \
  samples/422-122949-0031.wav sources/features/422-122949-0031.npy


## 7. (tuỳ chọn) Kiểm `dec` khớp PyTorch


In [ ]:
%%bash
set -e
cd /content/voice-rs
source $HOME/.cargo/env
cargo run --release -p voice-models --example verify_dec | tail -4


## 8. Huấn luyện adapter
Mặc định `SEG=86` (1 s) · `STEPS=250` · `LR=1e-3`. Tăng `SEG` để dùng cả câu
(589 frame) — chậm hơn ~7×.


In [ ]:
%%bash
set -e
cd /content/voice-rs
source $HOME/.cargo/env
SEG=86 STEPS=250 LR=1e-3 cargo run --release -p voice-train --example train_adapter 2>&1 | tee /content/train.log


## 9. Tải kết quả về máy


In [ ]:
from google.colab import files
files.download("/content/voice-rs/test_output.wav")
files.download("/content/voice-rs/sources/features/adapter.bpk")
